# Train the six-theme soft HMM

Plan: [plan.md](plan.md). GPL whitened windows, then panMP motif scores, then Baum–Welch on [song_soft_hmm_t6.dot](song_soft_hmm_t6.dot).

The 10 kHz recording is the path `WAV_PATH` in [run_train.py](run_train.py): `C:\Users\hoang\Downloads\SOCAL_H_76_230426_193500_df100.x.x_10kHz.wav`. GPL for that file is already under `work/gpl_193500/`. The command that produced it is:

```bash
python -m analysis.GPL_Python.process_v4 "C:\Users\hoang\Downloads\SOCAL_H_76_230426_193500_df100.x.x_10kHz.wav" --output-dir analysis/HMM/work/gpl_193500 --freq-lo 10 --freq-hi 1000
```

`main` reuses windows and motif scores already on disk.

In [ ]:
from analysis.HMM.run_train import main

main()

## Fit metrics

Sequence log-likelihood, nats per second against a standard-normal baseline, posterior mass on the labeled theme, and the per-song rates. The figures are written under `work/figures/`.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

from analysis.HMM.run_train import MODEL_PATH, WORK, load_model

report = json.loads((WORK / "train_report.json").read_text(encoding="utf-8"))
model, _bundle = load_model(MODEL_PATH)
print(f"songs {report['n_songs']}  motifs {report['n_motifs']}  seconds {report['n_seconds']}")
print("log-likelihood by iteration:")
for i, value in enumerate(report["loglik_history"], start=1):
    print(f"  {i:2d}  {value:.3f}")
print(f"saved log-likelihood {report['loglik']:.3f}  ({report['loglik_per_second']:.3f} nats/s)")
print(f"standard normal {report['normal_per_second']:.3f} nats/s")
print(f"initialization {report['initial_loglik']:.3f}")
print(
    "complete-data emission "
    f"{report['emission_complete']:.3f}, transition {report['transition_complete']:.3f}, "
    f"start {report['start_complete']:.3f}"
)
print(f"posterior mass on the labeled theme {report['posterior_on_label']:.3f}")
print(f"repeat probabilities {np.array2string(model.repeat_prob, precision=3)}")
print(f"start probabilities {np.array2string(model.start_prob, precision=3)}")

for name in ("loglik.png", "song_loglik.png", "theme_occupancy.png", "branch_sensitivity.png"):
    path = WORK / "figures" / name
    image = plt.imread(path)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(name)
    plt.show()
